# 19 · Chromatic dispersion

In a fiber, the group velocity depends on the wavelength.  A signal of
bandwidth `B` occupies a range of wavelengths, its spectral components
arrive at different times, and each symbol smears over its neighbours.
Over a long link, **chromatic dispersion** (CD) spreads a symbol over
hundreds of symbol periods: by far the largest linear impairment of a
coherent link, and one that a coherent receiver removes almost exactly in
DSP.

1. the size of the effect: the dispersion parameter `D` and the spread;
2. the all-pass transfer function: dispersion changes the phase, not the
   power spectrum;
3. electronic **compensation** (CDC) of a capture, and of a simulated
   periodic record;
4. the compensating **FIR filter**: its taps, its response and how many
   taps it needs;
5. **residual** dispersion, left to the adaptive equalizer.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)
RS = 32e9
D = 17.0  # ps/(nm km), standard single-mode fiber at 1550 nm
LAMBDA = 1550.0  # nm
C_LIGHT = 299_792_458.0


def host(a):
    return to_device(a, "cpu")


tx = ck.generate(
    C16,
    num_symbols=2**15,
    symbol_rate=RS,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=1,
    device=DEVICE,
)


def cd(x, km):
    return ck.impairments.apply_chromatic_dispersion(
        x, dispersion_ps_nm_km=D, fiber_length_km=km, center_wavelength_nm=LAMBDA
    )


def cdc(x, km, **kw):
    # The receiver knows the link's accumulated dispersion D·L in ps/nm.
    return ck.filtering.correct_chromatic_dispersion(
        x, dispersion_ps_nm=D * km, center_wavelength_nm=LAMBDA, **kw
    )


def snr_after(x):
    return ck.metrics.snr(
        ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(x))
    )

## 1. How large is it?

The dispersion parameter `D` (17 ps/(nm·km) for standard fiber at
1550 nm) is the group-delay difference per nm of wavelength and per km.
A signal of bandwidth `B` spans `Δλ = λ² B / c`, so after a length `L` its
components are spread over

$$\Delta\tau = |D|\, L\, \Delta\lambda = |D|\, L\, \frac{\lambda^2 B}{c}.$$

In [ ]:
bw = (1 + 0.1) * RS
dlam_nm = (LAMBDA * 1e-9) ** 2 * bw / C_LIGHT * 1e9
print(f"bandwidth {bw / 1e9:.1f} GHz -> Δλ = {dlam_nm:.3f} nm")
for km in (1, 80, 1000):
    spread = D * km * dlam_nm * 1e-12
    print(
        f"L = {km:5d} km: spread {spread * 1e9:7.3f} ns = {spread * RS:7.1f} symbol periods"
    )

## 2. An all-pass filter

In the frequency domain, dispersion multiplies the spectrum by a phase
that is quadratic in frequency,

$$H(f) = \exp\!\left(-j\, \frac{\beta_2 L}{2} (2\pi f)^2\right), \qquad
\beta_2 = -\frac{D \lambda^2}{2\pi c},$$

with `|H(f)| = 1`: the power spectrum is unchanged and no energy is lost.
Everything happens in the phase, which is why CD is invisible on a power
spectrum and yet destroys the waveform.

In [ ]:
rx80 = cd(tx, 80)
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_psd(tx, nperseg=1024, ax=axes[0], title=None, label="transmitted")
ck.plotting.plot_psd(
    rx80,
    nperseg=1024,
    ax=axes[0],
    title="Power spectrum: unchanged",
    label="after 80 km",
)
axes[0].legend()
# One isolated pulse: transmit a single symbol and watch it spread.
pulse = tx.xp.zeros(4096, dtype=tx.samples.dtype)
pulse[2048] = 1
shaped = ck.filtering.fir_filter(pulse, tx.pulse.taps(2))
t = (np.arange(4096) - 2048) / 2  # symbol periods
for km in (0, 20, 80):
    y = (
        shaped
        if km == 0
        else ck.impairments.apply_chromatic_dispersion(
            shaped,
            dispersion_ps_nm_km=D,
            fiber_length_km=km,
            center_wavelength_nm=LAMBDA,
            sampling_rate=2 * RS,
        )
    )
    axes[1].plot(
        t, np.abs(host(y)) ** 2 / np.max(np.abs(host(shaped)) ** 2), label=f"{km} km"
    )
axes[1].set_xlim(-15, 15)
axes[1].set_xlabel("Time [symbol periods]")
axes[1].set_ylabel("|pulse|² (normalized)")
axes[1].set_title("One RRC pulse")
axes[1].legend()
plt.show()

for km in (1, 5, 20, 80):
    print(f"{km:3d} km uncompensated: SNR {snr_after(cd(tx, km)):6.2f} dB (noise-free)")

## 3. Compensation

Because `H(f)` is known and all-pass, its inverse is exact:
`correct_chromatic_dispersion` filters with `H*(f)`.  It needs the
link's **accumulated dispersion** `D·L` in ps/nm, the number a receiver
knows from the link plan or measures, not the fiber's `D` and length
apart: only their product enters `β₂L`.

How it filters depends on what the record is:

* **a capture** (the default): a window of an endless stream, as an ADC
  delivers it.  The compensation is an FIR filter applied by overlap-save
  (section 4 designs it), and the first and last half filter of the
  window are transients, as in any receiver;
* **one period of a periodic sequence** (`periodic=True`): an AWG
  pattern played in a loop, or a record dispersed by
  `apply_chromatic_dispersion`, which treats its record as one period and
  disperses it circularly.  The response then multiplies the whole
  record's spectrum: exact on such a record, with no transients.

The simulated records of this notebook are periodic, so the whole-record
compensation inverts them exactly.  The constellations of the 80 km
link, both after the same matched filter and slicer: uncompensated, each
symbol is the sum of a dozen neighbours and the samples form one
Gaussian-looking cloud; compensated, the grid is back.

In [ ]:
for km in (80, 1000):
    rx = cd(tx, km)
    print(
        f"{km:5d} km: before {snr_after(rx):7.2f} dB, "
        f"after CDC {snr_after(cdc(rx, km, periodic=True)):6.2f} dB"
    )

rx80 = cd(tx, 80)
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
views = (("80 km, uncompensated", rx80), ("after CDC", cdc(rx80, 80, periodic=True)))
for ax, (name, field) in zip(axes, views, strict=True):
    symbols = ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(field))
    ck.plotting.plot_constellation(symbols, ax=ax, title=name)
plt.show()

A capture is different: its first samples carry the dispersed tails of
symbols sent before the capture started, and its last samples lack the
tails of symbols sent after.  Here a window from the middle of the
dispersed record stands in for a capture.  Within about half the spread
of its ends **no** compensation can be right, as the history it needs
lies outside the capture: as a capture (the default FIR) those samples
are filter transients, treated as periodic the window's other end wraps
in.  Beyond them both work: the FIR at its design accuracy of about
-45 dB, the whole-record filter with the exact response even better.
The FIR is the receiver's form: bounded memory on a stream of any length
(overlap-save), and taps to inspect.

In [ ]:
start, width = 2**14, 2**14  # samples
capture = rx80.samples[start : start + width]
sent = tx.samples[start : start + width]
kw = {"sampling_rate": tx.sampling_rate}
as_capture = cdc(capture, 80, bandwidth=(1 + 0.1) * RS, **kw)
as_periodic = cdc(capture, 80, periodic=True, **kw)

fig, ax = plt.subplots()
for name, y in (("as a capture (FIR)", as_capture), ("as periodic", as_periodic)):
    err = np.abs(host(y - sent)) ** 2 / np.mean(np.abs(host(sent)) ** 2)
    ax.semilogy(np.arange(200), err[:200] + 1e-12, label=name)
ax.set_xlabel("Sample of the capture")
ax.set_ylabel("Squared error (normalized)")
ax.set_title("The first 200 samples of a compensated capture")
ax.legend()
plt.show()

Two details:

* the sign convention: applying the compensation of the wrong sign doubles
  the dispersion instead of removing it, so the parameters are those of the
  fiber, for both the channel and the compensation;
* `center_wavelength_nm` matters because `β₂ ∝ D λ²`.

## 4. The compensating FIR filter

The FIR filter of a capture is a design choice: how many taps, fitted
over which band.  `correct_chromatic_dispersion` designs it with
`dispersion_compensation_taps`, which also returns the taps to inspect
or apply yourself: the least-squares fit of `H*(f)` over a band, by
default the whole sampled band.

The filter must hold the dispersion's memory.  Its group delay
`τ(f) = -β₂ L · 2πf` changes linearly across the band, so over a band
`B'` it spans `2π |β₂ L| B'` seconds: that **spread** in samples is the
minimum length.  Over the whole sampled band (`B' = fs`) it is the
classic estimate of Savory (2008); over the occupied band only, a
1.1 × 32 GHz = 35.2 GHz RRC signal at `fs` = 64 GS/s, it is 45 % shorter,
because the filter need not be right where the signal has no power.  The
default length is 1.25 times the spread; `bandwidth=` and `num_taps=`
pass straight through `correct_chromatic_dispersion`.

In [ ]:
FS = tx.sampling_rate
BAND = (1 + 0.1) * RS  # occupied bandwidth of the RRC signal
fiber = {"center_wavelength_nm": LAMBDA, "sampling_rate": FS}

for km in (80, 1000):
    whole = ck.filtering.dispersion_compensation_taps(dispersion_ps_nm=D * km, **fiber)
    band = ck.filtering.dispersion_compensation_taps(
        dispersion_ps_nm=D * km, bandwidth=BAND, **fiber
    )
    print(
        f"{km:5d} km: {len(whole):5d} taps over the whole band, {len(band):5d} over the occupied band"
    )

taps = ck.filtering.dispersion_compensation_taps(
    dispersion_ps_nm=D * 80, bandwidth=BAND, **fiber
)
response = ck.filtering.filter_response(taps, sampling_rate=FS, nfft=1024)
f = response.frequencies
centre = (len(taps) - 1) // 2 / FS  # the taps are centred when applied
beta2_l = -D * 1e-6 * (LAMBDA * 1e-9) ** 2 / (2 * np.pi * C_LIGHT) * 80e3

fig, axes = plt.subplots(
    1, 3, figsize=np.multiply(plt.rcParams["figure.figsize"], (3, 1))
)
n = np.arange(len(taps)) - (len(taps) - 1) // 2
axes[0].plot(n, np.abs(taps), label="|h|")
axes[0].plot(n, taps.real, alpha=0.6, label="real")
axes[0].set_xlabel("Tap")
axes[0].set_title(f"{len(taps)} taps for 80 km: a chirp")
axes[0].legend()
axes[1].plot(f / 1e9, 20 * np.log10(np.abs(response.response)))
axes[1].axvspan(-BAND / 2e9, BAND / 2e9, alpha=0.15, label="occupied band")
axes[1].set_ylim(-10, 10)
axes[1].set_xlabel("Frequency [GHz]")
axes[1].set_ylabel("|H| [dB]")
axes[1].set_title("Magnitude: flat in the band")
axes[1].legend()
axes[2].plot(f / 1e9, (response.group_delay - centre) * 1e12, label="FIR")
axes[2].plot(
    f / 1e9, -beta2_l * 2 * np.pi * f * 1e12, "--", label=r"$-\beta_2 L \cdot 2\pi f$"
)
axes[2].axvspan(-BAND / 2e9, BAND / 2e9, alpha=0.15)
axes[2].set_ylim(-600, 600)
axes[2].set_xlabel("Frequency [GHz]")
axes[2].set_ylabel("Group delay [ps]")
axes[2].set_title("Group delay: linear in frequency")
axes[2].legend()
plt.show()

The magnitude is 0 dB across the occupied band, and the group delay
follows the straight line of the inverse fiber: the higher frequencies,
which the anomalous fiber advanced, are delayed by the same amount.
Outside the band the fit is free, and gives up: the gain there rises to
10 dB and more, which amplifies whatever noise lies outside the signal
band (a matched filter removes it afterwards).

How many taps are enough?  The SNR after the FIR against its length
(`correct_chromatic_dispersion` with `num_taps=`, the transients
trimmed), with the exact whole-record compensation as the reference; a filter shorter than the spread cannot
reach it, and a fit over the whole band needs about three times the
taps of one over the occupied band to get there.

In [ ]:
rx80 = cd(tx, 80)


def snr_fir(x, bandwidth, num_taps):
    y = ck.multirate.decimate_to_symbol_rate(
        ck.filtering.matched_filter(cdc(x, 80, bandwidth=bandwidth, num_taps=num_taps))
    )
    k = num_taps  # symbols: more than the transient of num_taps // 2 samples
    err = host(y.samples[k:-k] - y.reference.symbols[k:-k])
    return 10 * np.log10(1 / np.mean(np.abs(err) ** 2))


lengths = np.arange(9, 100, 4)
fig, ax = plt.subplots()
for label, bw in (("occupied band", BAND), ("whole band", None)):
    snrs = [snr_fir(rx80, bw, int(n)) for n in lengths]
    ax.plot(lengths, snrs, "o-", label=f"fit over the {label}")
ax.axhline(
    snr_after(cdc(rx80, 80, periodic=True)), color="C3", ls="--", label="whole record"
)
spread = 2 * np.pi * abs(beta2_l) * BAND * FS
ax.axvline(spread, color="gray", ls=":", label=f"spread over the band ({spread:.0f})")
ax.set_xlabel("Number of taps")
ax.set_ylabel("SNR [dB]")
ax.set_title("FIR compensation of 80 km")
ax.legend()
plt.show()

## 5. Residual dispersion

The link length (or the fiber's exact `D`) is rarely known to the last
kilometre.  A CDC for the nominal length leaves a small residual, which
the adaptive equalizer ([16](16-adaptive-equalization.ipynb)) absorbs if it has enough taps.
Here an 80 km link is compensated for 70 to 80 km, and a 15-tap LMS (7.5
symbols at 2 samples per symbol) cleans up the rest:

In [ ]:
rx = ck.impairments.apply_awgn(cd(tx, 80), esn0_db=20, rng=2)
print("CDC length   CDC only   CDC + 15-tap LMS")
for km in (80, 78, 75, 70):
    y = ck.filtering.matched_filter(cdc(rx, km, periodic=True))
    lms = ck.equalization.lms(
        y, num_taps=15, step_size=3e-3, training_symbols=y.reference.symbols[..., :4000]
    )
    print(
        f"{km:7d} km   {ck.metrics.snr(ck.multirate.decimate_to_symbol_rate(y)):7.2f} dB   "
        f"{ck.metrics.snr(lms.signal, num_skip_symbols=8000):9.2f} dB"
    )

**Summary**

* The spread is `|D| L λ² B / c`: about 12 symbol periods per 80 km at 32 GBd.
* CD is an all-pass filter: the power spectrum stays, the waveform does
  not.
* `correct_chromatic_dispersion` takes the accumulated dispersion `D·L`.
  On a capture (the default) it applies an FIR filter by overlap-save; on
  a periodic record (`periodic=True`) the exact whole-record inverse.  The
  adaptive equalizer handles the residual.
* The FIR needs at least the spread of the group delay over the band it
  must hold: `dispersion_compensation_taps` designs it, `filter_response`
  shows it, and fitting only the occupied band cuts the taps to a third.
* The fiber's nonlinearity couples with dispersion at high power; that is
  the next notebook.

**Next:** [20 · Nonlinear fiber and digital backpropagation](20-nonlinear-fiber-and-backpropagation.ipynb).